In [ ]:
import pandas  as pd
df = pd.read_csv("titanic_cleaned.csv")
df.head()
!python -m pip install -U scikit-learn

In [ ]:
Y = df["Survived"].copy()
X = df[["Pclass","Age","Fare","FamilySize","Title","Has_Cabin","Sex_male"]].copy()
X.head()

,Pclass,Age,Fare,FamilySize,Title,Has_Cabin,Sex_male
0,3,22.0,7.2500,2,Mr,0,True
1,1,38.0,71.2833,2,Mrs,1,False
2,3,26.0,7.9250,1,Miss,0,False
3,1,35.0,53.1000,2,Mrs,1,False
4,3,35.0,8.0500,1,Mr,0,True


In [ ]:
X["Sex_male"] = X["Sex_male"].astype(int)

In [ ]:
title_dummies = pd.get_dummies(df['Title'], drop_first=True, prefix='Title', dtype=int)
X = pd.concat([X, title_dummies], axis=1)
X.drop('Title', axis=1, inplace=True, errors='ignore')
X.head()

,Pclass,Age,Fare,FamilySize,Has_Cabin,Sex_male,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Title_the Countess
0,3,22.0,7.2500,2,0,1,0,1,0,0,0
1,1,38.0,71.2833,2,1,0,0,0,1,0,0
2,3,26.0,7.9250,1,0,0,1,0,0,0,0
3,1,35.0,53.1000,2,1,0,0,0,1,0,0
4,3,35.0,8.0500,1,0,1,0,1,0,0,0


In [ ]:
X['Title_Rare'] = X['Title_Rare'] + X['Title_the Countess']
X.drop('Title_the Countess', axis=1, inplace=True)
X.head()

,Pclass,Age,Fare,FamilySize,Has_Cabin,Sex_male,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,3,22.0,7.2500,2,0,1,0,1,0,0
1,1,38.0,71.2833,2,1,0,0,0,1,0
2,3,26.0,7.9250,1,0,0,1,0,0,0
3,1,35.0,53.1000,2,1,0,0,0,1,0
4,3,35.0,8.0500,1,0,1,0,1,0,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

X_train.shape, X_test.shape

((623, 10), (268, 10))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

accuracy_scores = cross_val_score(pipeline, X_train, y_train,
                                   cv=kfold, scoring='accuracy')
roc_scores = cross_val_score(pipeline, X_train, y_train,
                              cv=kfold, scoring='roc_auc')

print(f"CV Accuracy : {accuracy_scores.mean():.3f} ± {accuracy_scores.std():.3f}")
print(f"CV ROC-AUC  : {roc_scores.mean():.3f} ± {roc_scores.std():.3f}")

CV Accuracy : 0.838 ± 0.018
CV ROC-AUC  : 0.887 ± 0.032


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Fit on ALL of X_train now (not just CV folds)
pipeline.fit(X_train, y_train)

# Evaluate on the held-out test set - only time you touch X_test
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

              precision    recall  f1-score   support

           0       0.81      0.88      0.84       153
           1       0.81      0.72      0.76       115

    accuracy                           0.81       268
   macro avg       0.81      0.80      0.80       268
weighted avg       0.81      0.81      0.81       268

Test ROC-AUC: 0.843


In [ ]:
import joblib

joblib.dump(pipeline, 'titanic_pipeline.joblib')
print("Pipeline saved.")

Pipeline saved.


In [ ]:
new_data_df = pd.DataFrame({
    'Pclass': [3, 1, 3],
    'Age': [25.0, 45.0, 18.0],
    'Fare': [15.0, 80.0, 7.5],
    'FamilySize': [1, 2, 1],
    'Has_Cabin': [0, 1, 0],
    'Sex_male': [1, 0, 0],
    'Title_Miss': [0, 0, 1],
    'Title_Mr': [1, 0, 0],
    'Title_Mrs': [0, 1, 0],
    'Title_Rare': [0, 0, 0]
})

print("Sample new data for prediction:")
new_data_df.head()

Sample new data for prediction:


,Pclass,Age,Fare,FamilySize,Has_Cabin,Sex_male,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,3,25.0,15.0,1,0,1,0,1,0,0
1,1,45.0,80.0,2,1,0,0,0,1,0
2,3,18.0,7.5,1,0,0,1,0,0,0


In [ ]:
import joblib

# Load the saved pipeline
pipeline = joblib.load('titanic_pipeline.joblib')

# Make predictions on the new data
predictions = pipeline.predict(new_data_df)

print("Predictions for new data:")
print(predictions)

Predictions for new data:
[0 1 1]


In [ ]:
pipeline = joblib.load('titanic_pipeline.joblib')
survival_proba_all = pipeline.predict_proba(new_data_df)[:, 1]
print("Survival probabilities for new data:")
for i, prob in enumerate(survival_proba_all):
    print(f"Instance {i+1}: {prob:.1%}")

Survival probabilities for new data:
Instance 1: 7.5%
Instance 2: 96.6%
Instance 3: 68.6%


1.8.0
